# EU CRR RAG — GPU Dev Environment

Run on a **T4 GPU** runtime (Runtime → Change runtime type → T4 GPU).

This notebook covers:
- **Cells 0–4**: Setup (clone, deps, secrets, HTML files)
- **Cells 5–8**: Ingestion (EN + IT, ~5 min on T4)
- **Cell 9**: Smoke-test query
- **Cells 10–12**: Integration tests (run per-file)
- **Cell 13**: Ad-hoc queries

> Unit tests run automatically on every push via GitHub Actions — no GPU needed for those.

## 0 — Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then reconnect.')

## 1 — Clone the repo

In [ ]:
import os

REPO_DIR = "/content/eu-crr-rag"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/apolitano20/EU-CRR-RAG.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

## 2 — Install dependencies

Colab already ships with a CUDA-enabled PyTorch — do **not** reinstall it.
This cell only installs the project-specific packages (~2–5 min, BGE-M3 model is ~570 MB).

In [ ]:
# Install from requirements files (Colab's CUDA PyTorch is already correct — do NOT reinstall torch)
!pip install -q -r requirements.txt -r requirements-dev.txt
print('Install complete.')

## 3 — API keys

Set secrets via the 🔑 key icon in the left sidebar (recommended), then run this cell.
Alternatively, paste values directly — but do **not** save the notebook with keys in it.

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["QDRANT_URL"]     = userdata.get("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = userdata.get("QDRANT_API_KEY")
os.environ["USE_RERANKER"]   = "true"  # T4 has enough VRAM for BGE-M3 + reranker

with open(".env", "w") as f:
    for k in ("OPENAI_API_KEY", "QDRANT_URL", "QDRANT_API_KEY"):
        f.write(f"{k}={os.environ[k]}\n")
    f.write("USE_RERANKER=true\n")

print(".env written. USE_RERANKER=true (GPU mode).")

## 4 — Upload HTML files

Upload `crr_raw_en.html` and `crr_raw_ita.html` using the Colab file browser **or** run the cell below to upload interactively.

In [ ]:
from google.colab import files
import shutil, os

print("Select your HTML files (crr_raw_en.html and crr_raw_ita.html):")
uploaded = files.upload()

for fname in uploaded:
    dest = os.path.join(REPO_DIR, fname)
    shutil.move(fname, dest)
    print(f"Moved {fname} → {dest}")

for expected in ["crr_raw_en.html", "crr_raw_ita.html"]:
    path = os.path.join(REPO_DIR, expected)
    if os.path.exists(path):
        print(f"  ✓ {expected} ({os.path.getsize(path)/1e6:.1f} MB)")
    else:
        print(f"  ✗ {expected} NOT FOUND — upload it before continuing")

## 5 — Warm up BGE-M3

Downloads the model (~570 MB) on first run. Confirms it is running on the GPU.

In [ ]:
import torch
from FlagEmbedding import BGEM3FlagModel

print("Loading BGE-M3 (first run downloads ~570 MB)...")
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)

# FlagEmbedding 1.x doesn't reliably auto-place on GPU via constructor args.
# Explicitly move to CUDA if available.
if torch.cuda.is_available():
    model.model = model.model.to("cuda")

device = next(model.model.parameters()).device
print(f"Model loaded on: {device}")

if str(device) == "cpu":
    raise RuntimeError("Model is on CPU — check that T4 GPU runtime is active (Runtime → Change runtime type).")

test_out = model.encode(["Article 92 own funds requirements"], return_dense=True, return_sparse=True)
print(f"Dense embedding dim: {len(test_out['dense_vecs'][0])}")
print("BGE-M3 OK.")

## 6 — Ingest English

`--reset` wipes the entire Qdrant collection first. Expected: 745 documents.

In [ ]:
!python -m src.pipelines.ingest_pipeline \
    --reset \
    --language en \
    --file crr_raw_en.html

## 7 — Ingest Italian

No `--reset` — appends to the existing collection. Expected: +745 documents.

In [ ]:
!python -m src.pipelines.ingest_pipeline \
    --language it \
    --file crr_raw_ita.html

## 8 — Validate

Expected item count: **~1490** (745 EN + 745 IT, no sub-chunking).

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from src.indexing.vector_store import VectorStore

vs = VectorStore()
vs.connect()
count = vs.item_count
print(f"Items in Qdrant collection: {count}")

if count >= 1480:
    print("PASS — item count is in expected range (>= 1480)")
elif count >= 740:
    print("PARTIAL — only one language ingested; re-run Step 7 for Italian")
else:
    print(f"FAIL — unexpectedly low count ({count}); check logs above")

## 9 — Smoke-test a query

Verifies that the QueryEngine loads and returns a substantive answer citing article numbers.

In [ ]:
import os
from dotenv import load_dotenv; load_dotenv()

from src.indexing.vector_store import VectorStore
from src.indexing.index_builder import HierarchicalIndexer
from src.query.query_engine import QueryEngine

vector_store = VectorStore()
indexer = HierarchicalIndexer(vector_store=vector_store)
engine = QueryEngine(vector_store=vector_store, indexer=indexer, openai_api_key=os.environ["OPENAI_API_KEY"])
engine.load()
print(f"Engine ready. use_reranker={engine.use_reranker}")

result = engine.query("What are the own funds requirements under Article 92?", language="en")

print("=" * 60)
print(result.answer)
print("=" * 60)
print(f"\nSources: {len(result.sources)}")
for s in result.sources:
    m = s.get('metadata', {})
    print(f"  Article {m.get('article','?')} [{m.get('language','?')}]  score={s.get('score',0):.3f}  expanded={s.get('expanded')}")

## Done

The Qdrant collection is now populated with ~1490 items (EN + IT, article-level, no sub-chunking).
Your local API server will use this index on next start — no re-ingest needed locally.

In [ ]:
QUERY    = "What is the leverage ratio requirement under Article 429?"
LANGUAGE = "en"   # "en", "it", or None for cross-language

result = engine.query(QUERY, language=LANGUAGE)
print("=" * 60)
print(result.answer)
print("=" * 60)
for s in result.sources:
    m = s.get('metadata', {})
    print(f"  Article {m.get('article','?')} [{m.get('language','?')}]  score={s.get('score',0):.3f}")

---
## Ad-hoc Queries

Edit `QUERY` and `LANGUAGE` then run. Engine must be loaded from the smoke-test cell above.

In [ ]:
!pytest tests/integration/test_query_engine.py -v --tb=short

In [ ]:
!pytest tests/integration/test_ingest_pipeline.py -v --tb=short

In [ ]:
!pytest tests/integration/test_vector_store.py -v --tb=short

---
## Integration Tests

Run per-file — BGE-M3 segfaults if all three run in one pytest invocation.